In [1]:
import os
import numpy as np
import torch
from pretty_midi import PrettyMIDI, Instrument
from poly_dis.model import PolyDisVAE

In [2]:
folder_path = "../COMMUDataset/losses/"

In [3]:
files_by_role = {
    'accompaniment': {'min': ['commu01310-005.npz', 'commu01287-001.npz', 'commu02622-001.npz', 'commu10104-001.npz'],
                      'max': ['commu04212-004.npz', 'commu04212-007.npz', 'commu03128-007.npz', 'commu03950-003.npz']},
    'bass': {'min': ['commu10308-004.npz', 'commu10308-002.npz', 'commu10825-003.npz', 'commu10643-002.npz'],
             'max': ['commu09796-001.npz', 'commu10103-001.npz', 'commu09796-004.npz', 'commu11096-001.npz']},
    'main_melody': {'min': ['commu00788-001.npz', 'commu06083-005.npz', 'commu02617-004.npz', 'commu05918-005.npz'],
                    'max': ['commu03994-002.npz', 'commu03994-006.npz', 'commu03151-005.npz', 'commu02846-002.npz']},
    'pad': {'min': ['commu02966-002.npz', 'commu02876-004.npz', 'commu03113-004.npz', 'commu10077-003.npz'],
            'max': ['commu10236-003.npz', 'commu10236-002.npz', 'commu00951-002.npz', 'commu04582-006.npz']},
    'riff': {'min': ['commu03851-001.npz', 'commu11141-002.npz', 'commu09946-002.npz', 'commu09946-004.npz'],
             'max': ['commu00374-007.npz', 'commu00370-003.npz', 'commu00370-002.npz', 'commu00370-004.npz']},
    'sub_melody': {'min': ['commu05514-002.npz', 'commu05321-002.npz', 'commu02565-002.npz', 'commu04823-001.npz'],
                   'max': ['commu01630-004.npz', 'commu01630-005.npz', 'commu01630-007.npz', 'commu01630-003.npz']}
}

In [4]:
out_root = "./HigherLowerLosses"
os.makedirs(out_root, exist_ok=True)

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
polydis = PolyDisVAE.init_model(device=device)
polydis.load_model("./poly_dis/model_param/polydis-v1.pt")

In [7]:
def load_latents(npz_path):
    d = np.load(npz_path, allow_pickle=True)
    z_chd = torch.tensor(d["z_chd"], dtype=torch.float32, device=device).unsqueeze(0)  # [1, D]
    z_txt = torch.tensor(d["z_txt"], dtype=torch.float32, device=device).unsqueeze(0)  # [1, D]
    bpm = None
    if "bpm" in d:
        try:
            bpm = int(str(d["bpm"]))
        except Exception:
            bpm = None
    return z_chd, z_txt, (bpm if bpm is not None else 80)

In [8]:
def decode_to_midi(z_h, z_t, bpm, name, out_dir):
    # decode
    ptree = polydis.pnotree_decode(z_h, z_t).squeeze(0)
    if isinstance(ptree, torch.Tensor):
        ptree = ptree.detach().cpu().numpy()
    # notes
    notes = polydis.pnotree_to_notes(ptree, bpm=bpm, start=0.0)
    # midi
    m = PrettyMIDI()
    inst = Instrument(program=0, name=name.replace(".mid", ""))
    inst.notes = notes
    m.instruments.append(inst)
    os.makedirs(out_dir, exist_ok=True)
    m.write(os.path.join(out_dir, name))

In [9]:
for role, groups in files_by_role.items():
    for grp, file_list in groups.items():  # grp ∈ {'min','max'}
        out_dir = os.path.join(out_root, role, grp)
        for fname in file_list:
            npz_path = os.path.join(folder_path, fname)
            z_h, z_t, bpm = load_latents(npz_path)
            out_name = f"{role}_{grp}_{fname.replace('.npz','.mid')}"
            decode_to_midi(z_h, z_t, bpm, out_name, out_dir)
            print(f"Saved {os.path.join(out_dir, out_name)} (BPM={bpm})")

Saved ./HigherLowerLosses/accompaniment/min/accompaniment_min_commu01310-005.mid (BPM=135)
Saved ./HigherLowerLosses/accompaniment/min/accompaniment_min_commu01287-001.mid (BPM=135)
Saved ./HigherLowerLosses/accompaniment/min/accompaniment_min_commu02622-001.mid (BPM=120)
Saved ./HigherLowerLosses/accompaniment/min/accompaniment_min_commu10104-001.mid (BPM=140)
Saved ./HigherLowerLosses/accompaniment/max/accompaniment_max_commu04212-004.mid (BPM=90)
Saved ./HigherLowerLosses/accompaniment/max/accompaniment_max_commu04212-007.mid (BPM=90)
Saved ./HigherLowerLosses/accompaniment/max/accompaniment_max_commu03128-007.mid (BPM=90)
Saved ./HigherLowerLosses/accompaniment/max/accompaniment_max_commu03950-003.mid (BPM=90)
Saved ./HigherLowerLosses/bass/min/bass_min_commu10308-004.mid (BPM=110)
Saved ./HigherLowerLosses/bass/min/bass_min_commu10308-002.mid (BPM=110)
Saved ./HigherLowerLosses/bass/min/bass_min_commu10825-003.mid (BPM=60)
Saved ./HigherLowerLosses/bass/min/bass_min_commu10643-002